---
title: "Лекція 8. Від навчених представлень (learned representations) до фундаментальних моделей (foundation models): машинне навчання в сучасному процесі наукового відкриття"
description: "Машинне навчання та методи вилучення знань | PhD F3 «Комп'ютерні науки» | КрНУ ім. М. Остроградського"
author: "&copy; [Valeriy Sydorenko](https://www.linkedin.com/in/valeriy-sydorenko-6782279a/), 2026"
date: today
jupyter: python3
bibliography: mlkdd-references.bib

format:
  html:
    code-fold: show
    code-summary: "Показати код"
    code-overflow: wrap
    toc: true
    toc-depth: 4
    toc-location: right
    number-sections: false
    embed-resources: true
    smooth-scroll: true
---

## Мета лекції

Систематизувати сучасну парадигму машинного навчання як інструментарій наукового відкриття: від ручного конструювання ознак до навчених представлень, самокерованого та попереднього навчання, фундаментальних моделей, пошуково-генеративних систем і нейросимволічних підходів. Сформувати критерії відокремлення *гіпотези, згенерованої моделлю*, від *емпірично підтвердженого наукового знання*.

## Очікувані результати навчання

Після лекції аспірант зможе:

1. Розрізняти ручні ознаки, навчені представлення та контекстні векторні представлення (embeddings).
2. Пояснювати принципи самокерованого навчання, контрастивних задач та попереднього навчання.
3. Характеризувати можливості й обмеження фундаментальних і мультимодальних моделей.
4. Розрізняти параметричну пам'ять моделі, пошук документів, цитування джерел та доказову базу.
5. Проєктувати перевірюваний ланцюг *джерело -- твердження -- метод перевірки -- доказ*.
6. Визначати роль людини, нейросимволічних обмежень і незалежної валідації у науковому відкритті.

## План лекції

1. Еволюція подання даних у машинному навчанні.
2. Навчені представлення та геометрія embeddings.
3. Самокероване навчання та попереднє навчання.
4. Фундаментальні моделі, мультимодальність і великі мовні моделі.
5. Параметричні знання, пошук та RAG.
6. Галюцинації, походження доказів і епістемологічна надійність.
7. Гібридні та нейросимволічні підходи.
8. Наукове відкриття у взаємодії людини й ШІ.
9. Наскрізний синтез ML/KDD-дослідження та підготовка до ЛР8.

Основні джерела: [@bengio2013representation; @lecun2015deep; @bommasani2021foundation; @fayyad1996kdd].

# 1. Еволюція представлень: від ознак до моделей загального призначення

У традиційному ML дослідник конструює ознаки $x=\phi_{\mathrm{manual}}(d)$, а модель $f_\theta(x)$ вивчає залежність між ними та цільовою змінною. У навчанні представлень (representation learning) перетворення $h_\varphi(d)$ також оптимізується за даними [@bengio2013representation].

$$
\hat y=f_\theta(h_\varphi(d)).
$$

Це не усуває дослідницьких рішень: добір корпусу, архітектура, функція втрат, попередня обробка та протокол оцінювання залишаються джерелами індуктивних припущень.

```{mermaid}
%%{init: {'theme': 'base', 'themeVariables': {'fontSize':'18px','fontFamily':'Trebuchet MS, Verdana, Arial, sans-serif','primaryColor':'#E8F3FF','primaryBorderColor':'#3478A8','secondaryColor':'#FFF0C9','tertiaryColor':'#E7F6EC','lineColor':'#536878'}}}%%
flowchart LR
 A("Сирі дані") --> B("Ручні ознаки")
 B --> C("Класична ML-модель")
 A --> D("Навчене представлення")
 D --> E("Спеціалізована модель")
 D --> F("Попередньо навчена модель")
 F --> G("Адаптація / пошук / генерація")
```

| Підхід | Що задає дослідник | Що навчається | Типова межа |
|---|---|---|---|
| Ручні ознаки | Відображення $\phi$ | Прогнозна функція | Втрата інформації через інженерію ознак |
| Навчені представлення | Архітектура, задача, дані | Представлення й прогноз | Залежність від навчального сигналу |
| Попереднє навчання | Корпус, ціль, масштаб | Переносимі параметри | Зсув домену та приховані властивості корпусу |
| Фундаментальна модель | Протокол адаптації та використання | Базова модель і/або адаптери | Необґрунтоване перенесення компетентності |

# 2. Векторні представлення та їхня геометрія

*Векторне представлення (embedding)* -- відображення об'єкта в числовий простір:

$$
h_\varphi:\mathcal D\longrightarrow\mathbb R^d.
$$

Для двох ненульових векторів косинусна подібність:

$$
\operatorname{sim}(u,v)=\frac{u^\top v}{\|u\|_2\|v\|_2}.
$$

Висока подібність відображає геометрію, сформовану навчальною задачею; вона *не доводить* логічної еквівалентності, істинності тверджень чи причинного зв'язку.

## Ручний приклад

Нехай $u=(1,0)$, $v=(1,1)$, $w=(0,1)$. Тоді:

$$
\operatorname{sim}(u,v)=\frac{1}{\sqrt2}\approx0.707,
\qquad
\operatorname{sim}(u,w)=0.
$$

Отже, $v$ ближчий до $u$, ніж $w$, але це не говорить, *чому* таке представлення було отримане й чи корисна ця близькість для конкретного дослідницького питання.

## Контекстні та статичні представлення

Статичні embeddings призначають одиниці мови один вектор незалежно від контексту. Контекстні представлення залежать від оточення токена та параметрів моделі. Це важливо для полісемії, доменної термінології, перенесення між корпусами та багатомовного аналізу.

# 3. Самокероване навчання та попереднє навчання

*Самокероване навчання (self-supervised learning)* використовує навчальний сигнал, сконструйований із самих даних, наприклад маскування частини тексту або передбачення наступного токена.

Для авторегресійного мовного моделювання:

$$
\mathcal L_{\mathrm{LM}}(\theta)=-\sum_{t=1}^{T}\log P_\theta(w_t\mid w_1,\ldots,w_{t-1}).
$$

Мінімізація цієї втрати сприяє моделюванню статистичної структури текстів, але не є прямою оптимізацією істинності наукових тверджень.

*Контрастивне навчання (contrastive learning)* наближує представлення пов'язаних об'єктів і віддаляє непов'язані. Для позитивної пари $(i,j)$ типовий вигляд втрати:

$$
\ell_{i,j}=-\log
\frac{\exp(\operatorname{sim}(z_i,z_j)/\tau)}
{\sum_{k\ne i}\exp(\operatorname{sim}(z_i,z_k)/\tau)},
$$

де $\tau>0$ -- температурний параметр. Вибір позитивних/негативних пар фактично визначає, яку інваріантність навчається зберігати модель.

*Попереднє навчання (pretraining)* створює параметри для повторного використання; *доадаптація (fine-tuning)* змінює їх під цільову задачу; *навчання з використанням контексту (in-context learning)* змінює поведінку через вхідний контекст без обов'язкової зміни ваг.

# 4. Фундаментальні моделі та мультимодальність

*Фундаментальна модель (foundation model)* -- модель, навчена на широких даних і придатна до адаптації для багатьох задач [@bommasani2021foundation]. Великі мовні моделі (large language models, LLM) є важливим, але не єдиним класом фундаментальних моделей.

Мультимодальність (multimodality) означає спільну роботу з різними типами даних -- текстом, зображеннями, аудіо, відео, структурованими вимірюваннями. Об'єднання модальностей саме по собі не гарантує надійного причинного або наукового узагальнення.

```{mermaid}
%%{init: {'theme': 'base', 'themeVariables': {'fontSize':'18px','fontFamily':'Trebuchet MS, Verdana, Arial, sans-serif','primaryColor':'#E8F3FF','primaryBorderColor':'#3478A8','secondaryColor':'#FFF0C9','tertiaryColor':'#E7F6EC','lineColor':'#536878'}}}%%
flowchart TD
 A("Текст") --> D("Спільний або пов'язаний<br/>простір представлень")
 B("Зображення") --> D
 C("Сигнали / вимірювання") --> D
 D --> E("Пошук і зіставлення")
 D --> F("Генерація гіпотез")
 D --> G("Адаптація до задачі")
 E --> H("Незалежна перевірка")
 F --> H
 G --> H
```

Для наукового дослідження модель може бути: (1) джерелом ознак; (2) інструментом пошуку; (3) генератором кандидатних гіпотез; (4) інтерфейсом до інструментів; (5) допоміжним засобом підготовки протоколу. Жодна з цих ролей не наділяє її висновки статусом перевіреного знання.

# 5. Параметрична пам'ять, пошук і RAG

*Параметричні знання* -- інформація, статистично відображена в параметрах моделі після навчання. Вони не мають автоматичного механізму відтворення точного джерела кожного твердження.

*Пошуково-доповнена генерація (retrieval-augmented generation, RAG)* поєднує пошук у зовнішньому корпусі з генерацією відповіді на основі відібраного контексту.

$$
D_k=\operatorname{Retrieve}(q,\mathcal C,k),
\qquad
\hat a=G_\theta(q,D_k).
$$

Тут $q$ -- запит, $\mathcal C$ -- корпус, $D_k$ -- відібрані фрагменти, $G_\theta$ -- генератор. *Наявність фрагмента в контексті не гарантує, що відповідь правильно його відображає.*

```{mermaid}
%%{init: {'theme': 'base', 'themeVariables': {'fontSize':'18px','fontFamily':'Trebuchet MS, Verdana, Arial, sans-serif','primaryColor':'#E8F3FF','primaryBorderColor':'#3478A8','secondaryColor':'#FFF0C9','tertiaryColor':'#E7F6EC','lineColor':'#536878'}}}%%
flowchart LR
 A("Дослідницький запит") --> B("Пошук у корпусі")
 B --> C("Відібрані фрагменти")
 C --> D("Генерація відповіді")
 D --> E("Перевірка тверджень")
 C --> E
 E --> F("Доказовий реєстр")
```

## Чотири різні властивості

| Властивість | Питання |
|---|---|
| Релевантність пошуку | Чи знайдено документ, дотичний до питання? |
| Точність цитування | Чи відповідає посилання конкретному фрагменту? |
| Підтримка твердження | Чи справді фрагмент логічно підтверджує висновок? |
| Наукова достовірність | Чи є дизайн, дані та методи джерела достатніми для висновку? |

# 6. Навчальний експеримент: від подібності до доказової підтримки

Продемонструємо мінімальний *відтворюваний пошуковий компонент*, не викликаючи зовнішні API. Використаємо TF-IDF + SVD як навчальну апроксимацію низьковимірного текстового представлення. **Це не фундаментальна модель і не нейронні embeddings.** Мета -- показати різницю між семантично-подібнісним пошуком і перевіркою доказу.

## 6.1. Контрольований корпус

Кожен фрагмент має ідентифікатор. Частина містить коректні твердження, частина -- застереження. Тексти синтетичні, створені для навчання; їх не можна цитувати як наукові публікації.

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize

corpus = pd.DataFrame([
    {"id": "D01", "text": "Cross fitting trains nuisance models on other folds to reduce overfitting bias."},
    {"id": "D02", "text": "SHAP explains model predictions and does not by itself identify causal effects."},
    {"id": "D03", "text": "Propensity score overlap is required for stable observational causal effect estimation."},
    {"id": "D04", "text": "A high classification AUC alone does not establish scientific causality."},
    {"id": "D05", "text": "Foundation models can support hypothesis generation but generated claims require verification."},
    {"id": "D06", "text": "Retrieval augmented generation uses external documents to condition generated responses."},
    {"id": "D07", "text": "Clustering always produces a partition even when natural groups do not exist."},
    {"id": "D08", "text": "An explanation is not evidence of a causal mechanism without a causal research design."},
])
display(corpus)

,id,text
0,D01,Cross fitting trains nuisance models on other ...
1,D02,SHAP explains model predictions and does not b...
2,D03,Propensity score overlap is required for stabl...
3,D04,A high classification AUC alone does not estab...
4,D05,Foundation models can support hypothesis gener...
5,D06,Retrieval augmented generation uses external d...
6,D07,Clustering always produces a partition even wh...
7,D08,An explanation is not evidence of a causal mec...


## 6.2. Побудова та порівняння представлень

TF-IDF зважує терміни залежно від їхньої частоти в документі та корпусі. Зменшення розмірності методом усіченого SVD дає компактне лінійне представлення. На малому корпусі результати сильно залежать від складу документів і словника.

In [2]:
vectorizer = TfidfVectorizer(lowercase=True, stop_words="english", ngram_range=(1, 2))
X_sparse = vectorizer.fit_transform(corpus["text"])
n_components = min(5, min(X_sparse.shape) - 1)
svd = TruncatedSVD(n_components=n_components, random_state=42)
Z = normalize(svd.fit_transform(X_sparse))

query = "Does SHAP prove causal effects?"
q_tfidf = vectorizer.transform([query])
q_svd = normalize(svd.transform(q_tfidf))

cosine_tfidf = (X_sparse @ q_tfidf.T).toarray().ravel()
cosine_svd = (Z @ q_svd.T).ravel()
ranked = corpus.copy()
ranked["tfidf_similarity"] = cosine_tfidf
ranked["svd_similarity"] = cosine_svd
display(ranked.sort_values("tfidf_similarity", ascending=False).round(3))

,id,text,tfidf_similarity,svd_similarity
1,D02,SHAP explains model predictions and does not b...,0.545,0.997
7,D08,An explanation is not evidence of a causal mec...,0.141,0.690
3,D04,A high classification AUC alone does not estab...,0.096,0.645
2,D03,Propensity score overlap is required for stabl...,0.063,0.459
0,D01,Cross fitting trains nuisance models on other ...,0.000,-0.000
4,D05,Foundation models can support hypothesis gener...,0.000,0.000
5,D06,Retrieval augmented generation uses external d...,0.000,0.000
6,D07,Clustering always produces a partition even wh...,0.000,0.000


### Інтерпретація

Оцінка подібності відповідає на запитання *«Які фрагменти геометрично найближчі до запиту?»*. Вона **не відповідає** на запитання *«Чи є твердження істинним і чи підтверджене воно якісним дослідженням?»*.

У нашому прикладі коректна відповідь: *SHAP пояснює прогноз моделі, але сам по собі не встановлює причинний ефект*. Висновок ґрунтується на змісті фрагментів і методологічному розрізненні з лекцій 5--7, а не на числовому значенні cosine similarity.

## 6.3. Реєстр перевірки тверджень (claim-evidence ledger)

Запропонуємо мінімальний запис, придатний для лабораторної роботи 8:

| Поле | Зміст |
|---|---|
| `claim_id` | Унікальний ідентифікатор твердження |
| `claim` | Точне формулювання |
| `source_id` | Документ або дані, на які посилаємося |
| `evidence_span` | Конкретний фрагмент, таблиця або результат |
| `support_type` | Підтримує / суперечить / не визначено |
| `methodological_quality` | Якість дизайну та обмеження |
| `verification` | Яку незалежну перевірку виконано |
| `status` | Гіпотеза / частково підтверджено / підтверджено в заданих межах |

Для демонстрації створимо *навчальний* запис. Його статус не поширюється на реальну літературу.

In [3]:
ledger = pd.DataFrame([{
    "claim_id": "C01",
    "claim": "SHAP сам по собі не ідентифікує причинні ефекти",
    "source_id": "D02; D08",
    "evidence_span": "Фрагменти про model predictions та causal design",
    "support_type": "Підтримує методологічне розрізнення",
    "methodological_quality": "Навчальні фрагменти; не рецензовані джерела",
    "verification": "Зіставити з причинною постановкою лекцій 5--7",
    "status": "Методологічне твердження; потребує авторитетних джерел"
}])
display(ledger.T)

,0
claim_id,C01
claim,SHAP сам по собі не ідентифікує причинні ефекти
source_id,D02; D08
evidence_span,Фрагменти про model predictions та causal design
support_type,Підтримує методологічне розрізнення
methodological_quality,Навчальні фрагменти; не рецензовані джерела
verification,Зіставити з причинною постановкою лекцій 5--7
status,Методологічне твердження; потребує авторитетни...


# 7. Галюцинації та епістемологічна надійність

*Галюцинація (hallucination)* -- згенероване твердження, яке є необґрунтованим, помилковим або не підтримується доступними джерелами. Можливі типові прояви: вигадане посилання, неправильно приписаний результат, підміна кореляції причинністю, непомічена зміна одиниці аналізу.

Для наукового використання важливо розрізняти:

1. *Синтаксичну правдоподібність* -- текст виглядає академічно.
2. *Фактологічну коректність* -- окремі твердження відповідають перевірюваним фактам.
3. *Джерельну обґрунтованість* -- існують точні та релевантні джерела.
4. *Методологічну валідність* -- дизайн і припущення дозволяють саме цей висновок.
5. *Відтворюваність* -- незалежний дослідник може перевірити процедуру та результат.

**Жоден рівень не випливає автоматично з попереднього.** Наявність коректного посилання не робить слабкий експеримент переконливим доказом.

## Приклад методологічної помилки

> «Фундаментальна модель знайшла закономірність у текстах і пояснила її, отже виявлено причинний механізм».

Помилка полягає у змішуванні *генерації гіпотези*, *виявлення асоціації* та *ідентифікації причинного ефекту*. Коректна дослідницька процедура вимагає формалізації причинного питання, дизайну перевірки й оцінювання невизначеності [@pearl2009causality; @hernan2020causal].

# 8. Гібридні та нейросимволічні підходи

*Гібридний підхід* поєднує статистичні моделі з алгоритмічними, предметними або логічними процедурами. *Нейросимволічний підхід (neuro-symbolic AI)* інтегрує навчені представлення із символічними правилами, онтологіями, логічними обмеженнями чи процедурами виведення.

Приклад: LLM пропонує структуру причинного DAG, але дослідник перевіряє часовий порядок, типи змінних і відсутність циклів. Формальна перевірка може відхилити циклічний граф; однак ациклічність **не доводить правильності причинних стрілок**.

```{mermaid}
%%{init: {'theme': 'base', 'themeVariables': {'fontSize':'18px','fontFamily':'Trebuchet MS, Verdana, Arial, sans-serif','primaryColor':'#E8F3FF','primaryBorderColor':'#3478A8','secondaryColor':'#FFF0C9','tertiaryColor':'#E7F6EC','lineColor':'#536878'}}}%%
flowchart LR
 A("Модель: кандидатна<br/>гіпотеза / DAG") --> B("Символічні правила<br/>та обмеження")
 B --> C("Предметна експертиза")
 C --> D("Емпіричний дизайн")
 D --> E("Перевірка й<br/>оновлення гіпотези")
 E -.-> A
```

## Мінімальна формальна перевірка DAG

У навчальному прикладі не потрібна окрема бібліотека графів. Для скінченного набору ребер реалізуємо перевірку циклу топологічним сортуванням.

In [4]:
from collections import defaultdict, deque

def is_dag(edges):
    nodes = {v for edge in edges for v in edge}
    indeg = {v: 0 for v in nodes}
    adj = defaultdict(list)
    for u, v in edges:
        adj[u].append(v)
        indeg[v] += 1
    queue = deque([v for v in nodes if indeg[v] == 0])
    visited = 0
    while queue:
        u = queue.popleft()
        visited += 1
        for v in adj[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                queue.append(v)
    return visited == len(nodes)

candidate_ok = [("motivation", "treatment"),
                ("motivation", "outcome"),
                ("treatment", "outcome")]
candidate_cycle = candidate_ok + [("outcome", "motivation")]
print("Перший граф -- DAG:", is_dag(candidate_ok))
print("Другий граф -- DAG:", is_dag(candidate_cycle))

Перший граф -- DAG: True
Другий граф -- DAG: False


# 9. Людина і ШІ у процесі наукового відкриття

Модель може розширити простір кандидатних гіпотез, знайти схожі роботи, запропонувати змінні та процедури. Але остаточна відповідальність за причинні припущення, дизайн експерименту, достовірність джерел і публікаційне твердження належить досліднику.

| Етап | Потенційна роль ШІ | Обов'язкова людська перевірка |
|---|---|---|
| Постановка RQ | Альтернативні формулювання | Наукова новизна та операціоналізація |
| Огляд літератури | Пошук і тематичне групування | Існування джерел, точність атрибуції |
| Представлення даних | Embeddings, multimodal fusion | Зсув, витік, обмеження даних |
| Гіпотези | Генерація кандидатних пояснень | Фальсифікованість і альтернативи |
| Експеримент | Пропозиції дизайну та коду | Валідність, незалежність, статистичний протокол |
| Синтез | Чернетка інтерпретації | Повнота доказів, межі узагальнення |

# 10. Наскрізний синтез курсу: від даних до наукового твердження

Повернемося до принципу, сформульованого на початку курсу: **модель, закономірність і наукове знання -- різні об'єкти** [@fayyad1996kdd; @breiman2001two].

```{mermaid}
%%{init: {'theme': 'base', 'themeVariables': {'fontSize':'18px','fontFamily':'Trebuchet MS, Verdana, Arial, sans-serif','primaryColor':'#E8F3FF','primaryBorderColor':'#3478A8','secondaryColor':'#FFF0C9','tertiaryColor':'#E7F6EC','lineColor':'#536878'}}}%%
flowchart TD
 A("RQ та операціоналізація<br/>v0.1") --> B("Експериментальний<br/>протокол v0.2")
 B --> C("Аудит валідності<br/>v0.3")
 C --> D("Валідація структури<br/>v0.4")
 D --> E("Аудит пояснень<br/>v0.5")
 E --> F("Причинна постановка<br/>v0.6")
 F --> G("Оцінювання ефектів<br/>v0.7")
 G --> H("Синтез доказів<br/>v0.8")
 I("Foundation models:<br/>пошук, гіпотези, представлення") -.-> A
 I -.-> D
 I -.-> H
```

## Наукове твердження як перевірювана конструкція

Пропонуємо операційний шаблон:

> Для визначеної популяції та одиниці аналізу, за зафіксованого протоколу, метод дав результат із зазначеною невизначеністю; додаткові перевірки підтримують таку інтерпретацію; висновок не поширюється на невивчені домени, невиміряні причини або альтернативні втручання.

Це не універсальна формула «істини», а дисципліна обмеження висновку доступними доказами.

# 11. Підготовка до лабораторної роботи 8

Завершальна ЛР8 має точну назву за НРП: *«Синтез результатів ML/KDD-дослідження та обґрунтування наукового твердження»*.

Її підсумковий артефакт -- *Scientific Claim & Evidence Dossier v0.8*. Він має поєднати:

1. Дослідницьке питання, популяцію, одиницю аналізу.
2. Джерела даних, представлення та протокол експерименту.
3. Валідацію, невизначеність, робастність і обмеження.
4. Перевірку структури та пояснень.
5. Причинну постановку й оцінювання ефекту -- **лише якщо це обґрунтовано RQ та даними**.
6. Реєстр тверджень, джерел і доказів.
7. Межі застосовності, альтернативні пояснення й відкриті питання.
8. Відтворюваний набір артефактів для незалежної перевірки.

## Дискусійний приклад

У дослідженні раннього виявлення ботоподібної активності foundation model може створювати embeddings повідомлень і допомагати формулювати пояснення. Але рання класифікація, групування акаунтів, пояснення моделі та причинний вплив модерації -- це **чотири різні дослідницькі задачі**. Для кожної потрібні власні дані, метрики та умови валідності.

# Питання для самоконтролю

1. Чим ручне конструювання ознак відрізняється від навчання представлень?
2. Чому косинусна подібність embeddings не є мірою істинності твердження?
3. Як самокероване навчання формує навчальний сигнал?
4. Які обмеження має контрастивне навчання?
5. Що відрізняє фундаментальну модель від вузькоспеціалізованої?
6. Які можливості та ризики мультимодальних представлень?
7. Чим параметрична пам'ять відрізняється від зовнішнього джерела?
8. Які етапи RAG необхідно перевіряти окремо?
9. Чому релевантний пошук не гарантує доказової підтримки?
10. Що таке галюцинація у контексті наукового аналізу?
11. Як відрізнити гіпотезу від перевіреного наукового результату?
12. Що додають символічні правила до нейронних моделей?
13. Чому формально коректний DAG не доводить причинну валідність?
14. Які елементи має містити реєстр наукових тверджень?
15. Як інтегрувати результати ЛР1--ЛР7 в одне обґрунтоване наукове твердження?

# Висновки

Навчені представлення та фундаментальні моделі розширюють можливості машинного навчання у науковому дослідженні: від подання складних даних до пошуку, генерації гіпотез та підтримки дослідницького процесу. Водночас **наукове відкриття не тотожне генерації правдоподібного тексту чи високій подібності векторів**. Його основу становлять дослідницький дизайн, перевірювані джерела, незалежна валідація, аналіз невизначеності та відповідально сформульовані межі висновку.

# Література та ресурси

Основні джерела: [@bengio2013representation; @lecun2015deep; @bommasani2021foundation; @fayyad1996kdd; @breiman2001two; @pearl2009causality; @hernan2020causal].

Додаткові ресурси: [scikit-learn: TruncatedSVD](https://scikit-learn.org/stable/modules/decomposition.html#truncated-singular-value-decomposition-and-latent-semantic-analysis), [Stanford CRFM](https://crfm.stanford.edu/), [The Turing Way: Reproducible Research](https://the-turing-way.netlify.app/).

::: {#refs}
:::